In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # so 'from inference import ...' below finds inference.py at the repo root

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.calibration import CalibratedClassifierCV
import shap

In [ ]:
from inference import CATEGORICAL_MAPS

DROP_COLUMNS = ['person_age', 'person_gender', 'person_education', 'person_home_ownership',
                'loan_intent', 'cb_person_cred_hist_length']

def preprocess_input(raw_df):
    """
    Applies the exact preprocessing used to build the training data (column
    drops + categorical mapping), so any caller - training code today, a
    website backend later - transforms raw applicant data the same way.
    Expects raw_df to contain all of the original CSV's columns. The
    categorical mapping itself comes from inference.py so training and the
    website always agree on how 'Yes'/'No' fields get encoded.
    """
    df = raw_df.drop(columns=DROP_COLUMNS)
    for column, mapping in CATEGORICAL_MAPS.items():
        df[column] = df[column].map(mapping)
    return df

db = pd.read_csv('../data/Project_DB_loan_approval.csv')
db = db[db['person_age'] > 18]
train_db = preprocess_input(db)

y_label = train_db['loan_status']
x_features = train_db.copy().drop(['loan_status'], axis = 1)
x_train, x_test, y_train, y_test = train_test_split(x_features, y_label, test_size=0.2, random_state=42, stratify=y_label)
scaler = StandardScaler()

def find_best_svc_pipeline(x_train, y_train, C_range, cv=5):
    """
    Performs a hyperparameter search over C for an SVC (kernel='rbf') 
    using Cross-Validation, and returns the fitted Pipeline with the 
    best-performing C value.

    Args:
        x_train - the normalizetion is happening inside the pipline
        y_train - the correct labels
        C_range - checking C for each point
        cv - the number of "splits" of the data inside x_train to make the evaluation more relaible

    Returns:
        tuple: best_pipeline: Pipeline of x_train
                best_C: value C
                best_score: the "winning" mean Accuracy
    """
    best_score = 0
    best_C = None
    best_pipeline = None

    for c_val in C_range:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', SVC(kernel='rbf', C=c_val))
        ])
        scores = cross_val_score(pipe, x_train, y_train, cv=cv, scoring='accuracy')
        mean_score = np.mean(scores)
        print(f"Testing C={c_val} | Mean Accuracy: {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_C = c_val
            best_pipeline = pipe
            
    print("-" * 40)
    print(f"The winning parameter is C={best_C} with an accuracy of {best_score:.4f}")
    best_pipeline.fit(x_train, y_train)
    y_pred_best = best_pipeline.predict(x_test)
    cm = confusion_matrix(y_test,y_pred_best)
    print("\nAccuracy:\n", accuracy_score(y_test, y_pred_best))
    print("Confusion matrix:\n", cm)
    print("Classification Report:\n")
    print(classification_report(y_test, y_pred_best))
    return best_pipeline, best_C, best_score
best_pipeline, best_C, best_score = find_best_svc_pipeline(x_train, y_train, C_range=[0.1, 1, 10])

In [ ]:
# The grid search above uses plain SVC (fast) to pick best_C. For deployment we
# need calibrated probabilities (for a confidence score on the website), so we
# refit the winning C with CalibratedClassifierCV. SVC's own `probability=True`
# is deprecated as of scikit-learn 1.9 in favor of this wrapper.
deployment_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CalibratedClassifierCV(SVC(kernel='rbf', C=best_C, random_state=42), ensemble=False))
])
deployment_pipeline.fit(x_train, y_train)
deployment_accuracy = accuracy_score(y_test, deployment_pipeline.predict(x_test))
print(f"Deployment (calibrated) pipeline test accuracy: {deployment_accuracy:.4f}")

In [ ]:
import joblib
joblib.dump(deployment_pipeline, '../model_artifacts/Full project.pkl')
print("File saved!")

In [ ]:
import hashlib
import json
from datetime import datetime

import shap

from inference import evaluate_application

def save_model_report(raw_pipeline, deployment_pipeline, model_name, features, x_train, x_test, y_test,
                       best_C, cv_accuracy, deployment_accuracy, csv_path, sample_size=10,
                       json_path='../model_artifacts/model_report.json'):
    """
    Saves a full report of the trained model: the feature names and the raw
    input schema/valid ranges a website form needs, a sample of the training
    data, the decision-function margins on the test set, the model's
    accuracy, its name, and a version stamp tying the model to the data it
    was trained on.

    Args:
        raw_pipeline - the uncalibrated Pipeline (has decision_function, used for margins)
        deployment_pipeline - the calibrated Pipeline used for confidence scores
        model_name - human-readable name of the model
        features - the feature columns used for training
        x_train, x_test, y_test - data used to build the sample and compute margins/accuracy
        best_C - the chosen C hyperparameter
        cv_accuracy - the cross-validation accuracy found during the search
        deployment_accuracy - test accuracy of the calibrated deployment pipeline
        csv_path - path to the raw training CSV, hashed for the version stamp
        sample_size - how many rows of x_train to store as a data sample
        json_path - where to save the human-readable report

    Returns:
        dict: the full report that was saved
    """
    y_pred = raw_pipeline.predict(x_test)
    margins = raw_pipeline.decision_function(x_test)
    cm = confusion_matrix(y_test, y_pred)

    # Realistic bounds a website form should validate against. Derived from
    # the training data, except credit_score which has a fixed, well-known
    # range (300-850) regardless of what happens to appear in this sample.
    valid_ranges = {col: (float(x_train[col].min()), float(x_train[col].max())) for col in features}
    if 'credit_score' in valid_ranges:
        valid_ranges['credit_score'] = (300, 850)

    with open(csv_path, 'rb') as f:
        training_data_hash = hashlib.sha256(f.read()).hexdigest()

    report = {
        'model_name': model_name,
        'best_C': best_C,
        'features': list(features),
        'input_schema': {col: str(x_train[col].dtype) for col in features},
        'categorical_mappings': CATEGORICAL_MAPS,
        'valid_ranges': valid_ranges,
        'data_sample': x_train.sample(n=min(sample_size, len(x_train)), random_state=42).to_dict(orient='records'),
        'margins': {
            'values': margins.tolist(),
            'min': float(margins.min()),
            'max': float(margins.max()),
            'mean': float(margins.mean()),
            'std': float(margins.std()),
        },
        'cv_accuracy': float(cv_accuracy),
        'test_accuracy': float(accuracy_score(y_test, y_pred)),
        'deployment_accuracy': float(deployment_accuracy),
        'confusion_matrix': cm.tolist(),
        'classification_report': classification_report(y_test, y_pred, output_dict=True),
        'training_data_hash': training_data_hash,
        'n_training_rows': len(x_train),
        'timestamp': datetime.now().isoformat(),
    }

    with open(json_path, 'w') as f:
        json.dump(report, f, indent=2)

    print(f"Model report saved to '{json_path}'")
    return report

model_name = (f"{type(best_pipeline.named_steps['model']).__name__} (kernel="
              f"{best_pipeline.named_steps['model'].kernel}), calibrated via CalibratedClassifierCV for deployment")
model_report = save_model_report(best_pipeline, deployment_pipeline, model_name, x_features.columns,
                                  x_train, x_test, y_test, best_C, best_score, deployment_accuracy,
                                  csv_path='../data/Project_DB_loan_approval.csv')

# A small reference sample for SHAP, saved once so a website backend doesn't
# need the full training set in memory just to explain a prediction.
background_sample = shap.sample(x_train, min(100, len(x_train)), random_state=42)
joblib.dump(background_sample, '../model_artifacts/background_sample.pkl')

# Demo: run one applicant from the raw dataset through the full pipeline above,
# using the same evaluate_application() a website backend would call.
demo_applicant = {col: db.iloc[0][col] for col in x_features.columns}
demo_result = evaluate_application(deployment_pipeline, demo_applicant, background_sample, model_report)
print("Demo prediction for one applicant:", demo_result)